# Spark structured streaming and Kafka - hello world

In [1]:
%run_nb spark-start

Args: Namespace(data_format='none', port_offset=2) - unknown_args: []
Spark version: 4.1.3, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: None
Spark packages: org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3
Spark extensions: 
Spark catalog configs: {}
spark.sql.shuffle.partitions: 200
spark.sparkContext.master: local[2]


Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcatalog
spark_catalog


Version,4.1.3
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        38Gi       5.9Gi        17Gi        35Gi        24Gi
Swap:          8.0Gi       337Mi       7.7Gi


# Set Spark spark.sql.shuffle.partitions
**For Structured Streaming queries, the number of shuffle partitions usually needs to be set much lower than for most batch queries—dividing the computation too much increases overheads and reduces throughput.**

In [2]:
print(f" Current: {spark.conf.get("spark.sql.shuffle.partitions")}")
spark.conf.set("spark.sql.shuffle.partitions", 20)
print(f" New: {spark.conf.get("spark.sql.shuffle.partitions")}")

 Current: 200
 New: 20


In [3]:
!kafkactl get topics

TOPIC        PARTITIONS     REPLICATION FACTOR
my-topic     1              1


# Create Kafka topic

In [4]:
%%bash 
kafkactl delete topic my-topic
kafkactl create topic my-topic \
  --partitions 1 \
  --replication-factor 1

topic deleted: my-topic
topic created: my-topic


In [5]:
!kafkactl config view

contexts:
    default:
        brokers:
            - kafka-4:29092
current-context: default



# Spark streaming
 - http://kafka-ui.localhost:8080/

In [6]:
checkpoint_location = "/tmp/checkpoints/my-topic-console"

Clean checkpoint

In [7]:
!rm -rf  {checkpoint_location}

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

KAFKA_BOOTSTRAP_SERVERS = "kafka-4:29092"
KAFKA_TOPIC = "my-topic"

json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("temperature", DoubleType(), True),
])

kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

parsed_stream = (
    kafka_stream
    .select(
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("partition"),
        F.col("offset"),
        F.col("value").cast("string").alias("raw_json"),
    )
    .withColumn(
        "data",
        F.from_json(F.col("raw_json"), json_schema),
    )
    .select(
        "kafka_timestamp",
        "partition",
        "offset",
        "raw_json",
        "data.*",
    )
)

In [9]:
query = (
    parsed_stream.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("checkpointLocation", checkpoint_location) 
    .start()
)

In [10]:
print(query.isActive)
print(query.lastProgress)

True
{
    "id": "07ef7451-81e9-4341-9d61-ab37a48612b8",
    "runId": "1c669069-c167-4034-b28e-fb7f94b1a2f0",
    "name": null,
    "timestamp": "2026-07-28T15:29:13.472Z",
    "batchId": 0,
    "batchDuration": 616,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "addBatch": 247,
        "commitOffsets": 59,
        "getBatch": 8,
        "latestOffset": 220,
        "queryPlanning": 52,
        "triggerExecution": 613,
        "walCommit": 21
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[my-topic]]",
            "startOffset": null,
            "endOffset": {
                "my-topic": {
                    "0": 0
                }
            },
            "latestOffset": {
                "my-topic": {
                    "0": 0
                }
            },
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "processedR

# Read and write messages
 - Write messages using Kafbat UI: http://kafka-ui.localhost:8080/ui/clusters/local/all-topics/my-topic
    - ```json
      {
          "name": "Bob",
          "age": 41,
          "city": "Sao Paulo",
          "temperature": 22.8
      }
      ```
 - Read messages on the console
   -  ```shell
      docker compose logs jupyter-spark-4.1 -f
      ```

In [14]:
print(query.isActive)
print(query.lastProgress)

True
{
    "id": "07ef7451-81e9-4341-9d61-ab37a48612b8",
    "runId": "1c669069-c167-4034-b28e-fb7f94b1a2f0",
    "name": null,
    "timestamp": "2026-07-28T15:30:24.130Z",
    "batchId": 1,
    "batchDuration": 0,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 0,
        "triggerExecution": 0
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[my-topic]]",
            "startOffset": {
                "my-topic": {
                    "0": 0
                }
            },
            "endOffset": {
                "my-topic": {
                    "0": 0
                }
            },
            "latestOffset": {
                "my-topic": {
                    "0": 0
                }
            },
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 0.0,
            "metrics": {
      

In [15]:
query.awaitTermination(1)  # waits at most 1 seconds

False

In [ ]:
query.awaitTermination()

In [13]:
!find  {checkpoint_location}
!test -f {checkpoint_location}/.metadata.crc && \
 hexdump {checkpoint_location}/.metadata.crc

/tmp/checkpoints/my-topic-console
/tmp/checkpoints/my-topic-console/metadata
/tmp/checkpoints/my-topic-console/.metadata.crc
/tmp/checkpoints/my-topic-console/offsets
/tmp/checkpoints/my-topic-console/offsets/0
/tmp/checkpoints/my-topic-console/offsets/.0.crc
/tmp/checkpoints/my-topic-console/commits
/tmp/checkpoints/my-topic-console/commits/0
/tmp/checkpoints/my-topic-console/commits/.0.crc
/tmp/checkpoints/my-topic-console/sources
/tmp/checkpoints/my-topic-console/sources/0
/tmp/checkpoints/my-topic-console/sources/0/0
/tmp/checkpoints/my-topic-console/sources/0/.0.crc
0000000 7263 0063 0000 0002 9737 9099          
000000c


In [ ]:
query.stop()